In [0]:
import time
from pyspark.sql import functions as F

source = spark.table("urban_mobility.silver.trip_events_clean")
zones = spark.table("urban_mobility.silver.zones")

total_rows = source.count()
print("Total available rows:", total_rows)

SIZES = {
    "100K": 100_000,
    "1M": 1_000_000,
    "5M": 5_000_000,
    "FULL": total_rows
}

results = []

for label, n in SIZES.items():
    fraction = min(1.0, n / total_rows)
    sample_df = source.sample(fraction=fraction, seed=42)
    sample_df.write.mode("overwrite").format("delta").saveAsTable(f"urban_mobility.monitoring.bench_{label.lower()}")
    actual_count = spark.table(f"urban_mobility.monitoring.bench_{label.lower()}").count()
    print(f"{label}: target={n}, actual={actual_count}")

In [0]:
import time
from pyspark.sql import functions as F

baseline_results = []

for label in ["100K", "1M", "5M", "FULL"]:
    df = spark.table(f"urban_mobility.monitoring.bench_{label.lower()}")

    start = time.time()
    result = (
        df.join(zones, df.pickup_zone_id == zones.zone_id, "inner")
        .groupBy("borough")
        .agg(F.count("*").alias("trip_count"))
    )
    result.count()
    elapsed = time.time() - start

    baseline_results.append({"size": label, "elapsed_sec": round(elapsed, 2)})
    print(f"Baseline {label}: {elapsed:.2f}s")

print(baseline_results)

In [0]:
optimized_results = []

for label in ["100K", "1M", "5M", "FULL"]:
    df = spark.table(f"urban_mobility.monitoring.bench_{label.lower()}")

    start = time.time()
    result = (
        df.join(F.broadcast(zones), df.pickup_zone_id == zones.zone_id, "inner")
        .groupBy("borough")
        .agg(F.count("*").alias("trip_count"))
    )
    result.count()
    elapsed = time.time() - start

    optimized_results.append({"size": label, "elapsed_sec": round(elapsed, 2)})
    print(f"Optimized (explicit broadcast) {label}: {elapsed:.2f}s")

print(optimized_results)

In [0]:
target_table = "urban_mobility.monitoring.bench_full"

start = time.time()
result = spark.table(target_table).filter(F.col("pickup_zone_id") == 237).count()
elapsed_before = time.time() - start
print(f"Before OPTIMIZE: {elapsed_before:.2f}s, rows found: {result}")

spark.sql(f"OPTIMIZE {target_table} ZORDER BY (pickup_zone_id)")

start = time.time()
result2 = spark.table(target_table).filter(F.col("pickup_zone_id") == 237).count()
elapsed_after = time.time() - start
print(f"After OPTIMIZE: {elapsed_after:.2f}s, rows found: {result2}")

improvement_pct = round((elapsed_before - elapsed_after) / elapsed_before * 100, 1)
print(f"Improvement: {improvement_pct}%")

In [0]:
target_table = "urban_mobility.monitoring.bench_full"
df = spark.table(target_table)

start = time.time()
result_baseline = df.groupBy("pickup_zone_id").agg(F.count("*").alias("cnt"))
result_baseline.count()
elapsed_baseline = time.time() - start
print(f"Baseline (default partitions): {elapsed_baseline:.2f}s")

start = time.time()
result_optimized = df.repartition(8, "pickup_zone_id").groupBy("pickup_zone_id").agg(F.count("*").alias("cnt"))
result_optimized.count()
elapsed_optimized = time.time() - start
print(f"Optimized (repartition by key, 8 partitions): {elapsed_optimized:.2f}s")

improvement_pct = round((elapsed_baseline - elapsed_optimized) / elapsed_baseline * 100, 1)
print(f"Improvement: {improvement_pct}%")